# MNIST Shared LoRA Pilot

Goal: test whether a single requirement-conditioned MNIST LoRA can replace one LoRA per requirement. This notebook prepares a small distillation-style training set from the existing RBT4DNN MNIST outputs, then evaluates generated images after you run training/generation in Colab.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/casparbreloh/rbt4dnn-seminar.git"
COLAB_ROOT = Path("/content/rbt4dnn-seminar")
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / "data/requirements.csv").exists()), None)
if ROOT is None:
    ROOT = COLAB_ROOT
    if (ROOT / ".git").exists():
        subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"], check=True)
    else:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)

for module in [
    "shared",
    "mnist",
    "precondition_validity",
    "cost_analysis",
    "mnist_lora_ablation",
    "shared_lora_pilot",
]:
    sys.modules.pop(module, None)

EXPERIMENT = ROOT / "experiments" / "mnist-shared-lora-pilot"
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(EXPERIMENT))
print(ROOT)
commit = subprocess.check_output(
    ["git", "-C", str(ROOT), "log", "-1", "--oneline"], text=True
).strip()
print(commit)

In [ ]:
from shared_lora_pilot import prepare_training_data, write_template

train_dir = prepare_training_data(ROOT, max_per_requirement=100)
template = write_template(ROOT)
print("training data:", train_dir)
print("result template:", template)
print("images:", len(list((train_dir / "images").glob("*.png"))))

## Colab training step

Use the generated folder `outputs/mnist-shared-lora-pilot/train/` as the captioned image dataset. Train one text-conditioned LoRA with all M1-M6 prompts mixed together. After generation, save images as:

```text
outputs/mnist-shared-lora-pilot/generated/M1/*.png
outputs/mnist-shared-lora-pilot/generated/M2/*.png
...
outputs/mnist-shared-lora-pilot/generated/M6/*.png
```

Generate the same count per requirement, ideally 100 or 1000. Then rerun the evaluation cell below.

In [ ]:
from pathlib import Path

print(
    "Training is intentionally left to the Colab LoRA tool you choose, because FLUX access and GPU memory vary."
)
print("Recommended output folder:", ROOT / "outputs" / "mnist-shared-lora-pilot" / "generated")
print("The next cell evaluates that folder once generated images exist.")

In [ ]:
from shared_lora_pilot import generated_folders, write_results

folders = generated_folders(ROOT)
missing = [req for req, folder in folders.items() if not list(folder.glob("*.png"))]
if missing:
    print("No generated images yet for:", ", ".join(missing))
else:
    results_path, summary_path = write_results(ROOT)
    print(results_path)
    print(summary_path)

This pilot is the retraining extension. It only becomes a result once generated images are written back and `results.csv` is produced.